# capsule：元画像・10回反復・edge回転拡張モデルのAUROC比較
比較学習ノートブックで完成した3モデルを、同じテスト132枚（正常23・異常109）で評価します。
正常のみの学習と、全テストでの評価を区別し、評価画像は回転・補完しません。全体を128×128白黒・0〜1にします。
全モデルで平均推論、SSIM窓11×11の異常度マップ(1−SSIM)の上位130点の平均を使用。独立した130枚のパッチではありません。
性能指標はAUROCのみで、閾値探索はしません。正常画像の復元SSE/MSEは補助情報です。

## 使い方
1. 比較学習ノートブックの3条件×100エポックを完了する。
2. 下の `RUN_ID` に完了時の実行IDを指定する。`None` の場合は、このcapsule比較の完了済み実行の最新を選び、実行IDを表示する。
3. 上から実行。結果と図は対象実行フォルダ内の新しい `evaluations/日時_ID/` に保存する。

**学習済みモデルがない間は、少量の動作確認だけを行います。未学習モデルの性能を実験結果として報告しません。**
単一シードの優劣と一般的な効果は区別し、bottleとの違いも形状だけの効果とは断定しません。

In [1]:
from pathlib import Path
from io import BytesIO
import hashlib
import json
import platform
import tempfile
import uuid
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision import transforms
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

%matplotlib inline

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'code/261021画像反復と回転拡張による学習効果の比較.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('このリポジトリ内を作業ディレクトリにしてください。')
RUN_ID = None  # 学習完了時に表示された実行IDを指定。Noneは完了済みcapsule実行の最新を選択。
OUTPUT_BASE = ROOT / 'saved_models/capsule_rotation_control_20261022'
TEST_DIR = ROOT / 'data/capsule/test'
RUN_SMOKE_TEST = True
EXPERIMENT = 'capsule_rotation_control_20261022_edge'
SEED = 42
CONDITIONS = ('A_original', 'B_repeat10', 'C_rotate10')
LABELS = {'A_original': 'A：元画像', 'B_repeat10': 'B：10回反復', 'C_rotate10': 'C：edge回転拡張'}
COLORS = {'A_original': '#526b8e', 'B_repeat10': '#527f6c', 'C_rotate10': 'darkorange'}
TOP_K = 130
EVAL_BATCH_SIZE = 32
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
torch.use_deterministic_algorithms(True)

fonts = {v.name for v in font_manager.fontManager.ttflist}
for font in ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']:
    if font in fonts:
        plt.rcParams['font.family'] = font
        break
else:
    import japanize_matplotlib
plt.rcParams.update({'font.size': 11, 'axes.unicode_minus': False, 'figure.dpi': 100})

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


completed_runs=[]
if OUTPUT_BASE.is_dir():
    for d in sorted(OUTPUT_BASE.iterdir(),key=lambda p:p.name):
        if (d/'config.json').is_file():
            config=read_json(d/'config.json')
            if config.get('experiment')==EXPERIMENT and config.get('status')=='completed': completed_runs.append(d)
RUN_DIR=OUTPUT_BASE/RUN_ID if RUN_ID is not None else (completed_runs[-1] if completed_runs else None)
READY=RUN_DIR is not None
RESULT_DIR=None
if READY:
    RUN_ID=RUN_DIR.name
    train_config=read_json(RUN_DIR/'config.json')
    if train_config.get('status')!='completed' or train_config.get('experiment')!=EXPERIMENT:
        raise ValueError('このcapsule比較の完了済み実行を指定してください。')
    assert train_config['epochs']==100 and train_config['batch_size']==32
    assert train_config['rotation_background']=='edge' and train_config['rotation_stage']=='full_resolution_before_resize'
    assert train_config['rotation_input_size']==[1000,1000] and train_config['center_crop'] is False
    assert train_config['angles']==list(range(0,360,36))
    model_paths={name:RUN_DIR/f'seed_{SEED}'/name/'model_final.pth' for name in CONDITIONS}
    summaries={name:read_json(path.parent/'summary.json') for name,path in model_paths.items()}
    assert all(path.is_file() for path in model_paths.values())
    assert all(s['seed']==SEED and s['epochs']==100 for s in summaries.values())
    assert len({s['initial_state_sha256'] for s in summaries.values()})==1
    assert summaries['A_original']['updates']==700
    assert summaries['B_repeat10']['updates']==summaries['C_rotate10']['updates']==6900
    assert summaries['B_repeat10']['sample_order_sha256']==summaries['C_rotate10']['sample_order_sha256']
    before_model_hashes={name:sha256(path) for name,path in model_paths.items()}
    print(f'対象実行：{RUN_ID} / seed：{SEED} / 推論：平均 / device：{DEVICE}')
else:
    print('正式学習済みモデルはまだありません。動作確認のみ実行し、正式なAUROC・比較図は作成しません。')

正式学習済みモデルはまだありません。動作確認のみ実行し、正式なAUROC・比較図は作成しません。


## 1. モデル・AUROC・前処理の定義
ROCと台形面積をNumPyで求めます。完全分離・逆順位・同点の例に加え、実データでも正常異常の順位によるAUROCと照合します。

In [2]:
class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar

def binary_roc_curve(y_true, scores):
    labels = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or labels.shape != scores.shape:
        raise ValueError('ラベルとスコアは同じ長さの1次元配列にしてください。')
    if not np.isfinite(scores).all() or not np.isin(labels, [0, 1]).all():
        raise ValueError('スコアは有限値、ラベルは0または1である必要があります。')
    positives = int((labels == 1).sum())
    negatives = int((labels == 0).sum())
    if not positives or not negatives:
        raise ValueError('正常と異常の両方が必要です。')
    order = np.argsort(-scores, kind='stable')
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(ordered_scores)), len(scores) - 1]
    true_positives = np.cumsum(ordered_labels)[ends]
    false_positives = (ends + 1) - true_positives
    fpr = np.r_[0.0, false_positives / negatives]
    tpr = np.r_[0.0, true_positives / positives]
    thresholds = np.r_[np.inf, ordered_scores[ends]]
    return fpr, tpr, thresholds

def binary_auroc(y_true, scores):
    fpr, tpr, _ = binary_roc_curve(y_true, scores)
    return float(np.trapezoid(tpr, fpr))

assert binary_auroc([0,0,1,1], [0.1,0.2,0.8,0.9]) == 1.0
assert binary_auroc([0,0,1,1], [0.9,0.8,0.2,0.1]) == 0.0
assert binary_auroc([0,0,1,1], [0.5,0.5,0.5,0.5]) == 0.5

transform = transforms.Compose([
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

## 2. 共通テスト画像とデータ識別情報
学習時に記録した正常画像・前処理の一致と、学習画像との完全一致がないことを正式評価前に確認します。

In [3]:
category_labels={'good':'正常','crack':'亀裂','faulty_imprint':'印字不良','poke':'穴・突き傷','scratch':'擦り傷','squeeze':'変形'}
paths = sorted(TEST_DIR.glob('*/*.png'), key=lambda p: p.relative_to(TEST_DIR).as_posix())
counts = {name: sum(p.parent.name == name for p in paths) for name in category_labels}
if counts != {'good':23,'crack':23,'faulty_imprint':22,'poke':21,'scratch':23,'squeeze':20}:
    raise ValueError(f'計画時の評価データ132枚と異なります: {counts}')

inputs, image_records = [], []
for path in paths:
    raw = path.read_bytes()
    with Image.open(BytesIO(raw)) as im:
        tensor = transform(im.convert('RGB'))
    inputs.append(tensor)
    image_records.append({
        'relative_path': path.relative_to(TEST_DIR).as_posix(),
        'repo_path': path.relative_to(ROOT).as_posix(),
        'category': path.parent.name,
        'category_ja': category_labels[path.parent.name],
        'true_label': int(path.parent.name != 'good'),
        'file_sha256': hashlib.sha256(raw).hexdigest(),
        'preprocessed_sha256': hashlib.sha256(tensor.numpy().tobytes()).hexdigest(),
    })
input_tensor = torch.stack(inputs)
image_table = pd.DataFrame(image_records)
assert tuple(input_tensor.shape) == (132,1,128,128)

if READY:
    normal_manifest = {r['path']:r for r in read_json(RUN_DIR / 'normal_test_manifest.json')}
    current_normal = [r for r in image_records if r['category']=='good']
    assert {r['repo_path'] for r in current_normal} == set(normal_manifest)
    for row in current_normal:
        old = normal_manifest[row['repo_path']]
        assert row['file_sha256'] == old['file_sha256'], row['relative_path']
        assert row['preprocessed_sha256'] == old['preprocessed_sha256'], row['relative_path']
    train_manifest = read_json(RUN_DIR / 'train_manifest.json')
    train_raw_hashes = {r['file_sha256'] for r in train_manifest}
    train_pixel_hashes = {r['preprocessed_sha256'] for r in train_manifest}
    assert not train_raw_hashes & {r['file_sha256'] for r in image_records}
    assert not train_pixel_hashes & {r['preprocessed_sha256'] for r in image_records}

manifest_hash = hashlib.sha256(json.dumps(image_records, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
display(pd.DataFrame([{'種類': category_labels[name], '枚数': count} for name,count in counts.items()]))
print(f'評価データ識別値：{manifest_hash}')

,種類,枚数
0,正常,23
1,亀裂,23
2,印字不良,22
3,穴・突き傷,21
4,擦り傷,23
5,変形,20


評価データ識別値：b82c309cc1a2898fcb2ddcc2ab5de933bab7cab0b9125ce2e8ccfe97f17c8f80


## 3. 少量の動作確認と評価関数
未学習の一時重みを正常2枚・異常2枚で確認します。ここでの数値は性能として表示・保存しません。

In [4]:
def score_model(model_path,images,batch_size=EVAL_BATCH_SIZE):
    model=VAE_Hybrid_CNN_FC(latent_dim=64).to(DEVICE)
    model.load_state_dict(torch.load(model_path,map_location=DEVICE,weights_only=True),strict=True)
    model.eval()
    restored=np.empty((len(images),128,128),dtype=np.float32)
    maps=np.empty_like(restored)
    scores=np.empty(len(images),dtype=np.float64)
    errors=np.empty(len(images),dtype=np.float64)
    with torch.inference_mode():
        for start in range(0,len(images),batch_size):
            x=images[start:start+batch_size].to(DEVICE)
            recon,_,_=model(x,sample=False)
            assert torch.isfinite(recon).all()
            sse=(x-recon).square().flatten(1).sum(1).cpu().numpy()
            for local in range(len(x)):
                index=start+local
                original=x[local,0].cpu().numpy(); output=recon[local,0].cpu().numpy()
                _,smap=ssim(original,output,data_range=1.0,win_size=11,full=True)
                amap=1.0-smap
                assert np.isfinite(amap).all()
                restored[index]=output; maps[index]=amap
                scores[index]=np.mean(np.sort(amap.ravel())[::-1][:TOP_K])
                errors[index]=sse[local]
    del model
    return scores,{'reconstruction':restored,'anomaly_map':maps,'sse_per_image':errors}

def verify_auroc(y,scores):
    value=binary_auroc(y,scores)
    normal=scores[y==0]; abnormal=scores[y==1]
    rank=((abnormal[:,None]>normal).sum()+0.5*(abnormal[:,None]==normal).sum())/(len(normal)*len(abnormal))
    assert np.isclose(value,rank,atol=1e-12)
    return value

if RUN_SMOKE_TEST:
    normal_indices=np.flatnonzero(image_table.true_label.to_numpy()==0)[:2]
    abnormal_indices=np.flatnonzero(image_table.true_label.to_numpy()==1)[:2]
    small_indices=np.r_[normal_indices,abnormal_indices]
    with tempfile.TemporaryDirectory(prefix='capsule_eval_smoke_') as tmp:
        temp_path=Path(tmp)/'untrained_smoke_only.pth'
        torch.manual_seed(123)
        smoke_model=VAE_Hybrid_CNN_FC(latent_dim=64).eval()
        torch.save(smoke_model.state_dict(),temp_path)
        scores1,cache1=score_model(temp_path,input_tensor[small_indices],batch_size=2)
        scores2,cache2=score_model(temp_path,input_tensor[small_indices],batch_size=2)
        assert np.array_equal(scores1,scores2)
        assert np.array_equal(cache1['reconstruction'],cache2['reconstruction'])
        assert np.isfinite(scores1).all()
        verify_auroc(image_table.true_label.to_numpy()[small_indices],scores1)
        del smoke_model,cache1,cache2,scores1,scores2
    print('一時重みの保存・読み込み、平均推論の反復一致、SSIM Top-130、AUROCの順位との一致を確認。')
    print('未学習の動作確認用モデルのため、性能値は報告せず、正式結果にも保存しません。')

一時重みの保存・読み込み、平均推論の反復一致、SSIM Top-130、AUROCの順位との一致を確認。
未学習の動作確認用モデルのため、性能値は報告せず、正式結果にも保存しません。


## 4. 正式モデル3条件の評価
学習時に保存した共通正常画像の復元誤差との一致も確認します。

In [5]:
all_scores={}; reconstruction_cache={}; summary_rows=[]
comparison=pd.DataFrame()
if READY:
    for name in CONDITIONS:
        scores,cache=score_model(model_paths[name],input_tensor)
        y=image_table.true_label.to_numpy()
        auc=verify_auroc(y,scores)
        normal_sse=float(cache['sse_per_image'][y==0].mean())
        recorded=float(summaries[name]['normal_test_sse_per_image'])
        assert np.isclose(normal_sse,recorded,rtol=1e-5,atol=1e-4),(name,normal_sse,recorded)
        all_scores[name]=scores; reconstruction_cache[name]=cache
        summary_rows.append({'condition':name,'seed':SEED,'auroc':auc,
            'normal_test_sse_per_image':normal_sse,'normal_test_mse_per_pixel':normal_sse/(128*128),
            'final_train_sse_per_image':summaries[name]['final_train_sse_per_image'],
            'updates':summaries[name]['updates'],'model_sha256':sha256(model_paths[name])})
        print(f'{LABELS[name]}：AUROC={auc:.6f} / 共通正常画像SSE={normal_sse:.6f}')
    comparison=pd.DataFrame(summary_rows)
    RESULT_DIR=RUN_DIR/'evaluations'/(datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    RESULT_DIR.mkdir(parents=True,exist_ok=False)
else:
    print('正式学習後に実行すると、3モデルのAUROCと共通正常画像の復元誤差を比較します。')

正式学習後に実行すると、3モデルのAUROCと共通正常画像の復元誤差を比較します。


## 5. AUROCの比較表と棒グラフ

In [6]:
if READY:
    display(comparison[['condition','seed','updates','auroc',
                        'normal_test_sse_per_image','normal_test_mse_per_pixel',
                        'final_train_sse_per_image']].rename(columns={
        'condition':'条件','seed':'学習seed','updates':'更新回数','auroc':'AUROC',
        'normal_test_sse_per_image':'共通正常画像のSSE/画像',
        'normal_test_mse_per_pixel':'共通正常画像のMSE/画素',
        'final_train_sse_per_image':'最終エポックの学習SSE/画像',
    }).style.format({'AUROC':'{:.6f}','共通正常画像のSSE/画像':'{:.4f}',
                     '共通正常画像のMSE/画素':'{:.6f}','最終エポックの学習SSE/画像':'{:.4f}'}))

    fig, ax = plt.subplots(figsize=(9,5.5))
    values = comparison.set_index('condition').loc[list(CONDITIONS),'auroc'].to_numpy()
    ax.bar(np.arange(3),values,color=[COLORS[name] for name in CONDITIONS],width=0.55)
    ax.set_xticks(np.arange(3),[LABELS[name] for name in CONDITIONS])
    ax.set_ylim(0,1.08)
    ax.set_yticks(np.linspace(0,1,6))
    ax.set_ylabel('AUROC（大きいほどよい）',fontsize=14)
    ax.set_title('元画像・画像反復・回転拡張モデルのAUROC比較',fontsize=16,fontweight='bold')
    for i,value in enumerate(values):
        ax.text(i,value+0.02,f'{value:.4f}',ha='center',fontsize=14,fontweight='bold')
    ax.grid(axis='y',linestyle='--',alpha=0.5)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(RESULT_DIR/'auroc_comparison.png',dpi=150)
    plt.show()
    plt.close(fig)

    winner = comparison.loc[comparison.auroc.idxmax()]
    display(Markdown(f'今回の単一シードの比較で最も高いAUROCは **{LABELS[winner.condition]}：{winner.auroc:.4f}** です。'
                     '学習シードを変えたときのばらつきや、有意な優劣はまだ確認していません。'))
else:
    print('正式結果待ち：この比較表示はまだありません。')

正式結果待ち：この比較表示はまだありません。


## 6. ROC曲線
既存の書式に合わせ8×8・右下凡例・破線グリッド。重ね描きのため3条件を色分けし、Cはオレンジです。

In [7]:
if READY:
    fig, ax = plt.subplots(figsize=(8,8))
    for name in CONDITIONS:
        fpr,tpr,_ = binary_roc_curve(image_table.true_label.to_numpy(),all_scores[name])
        auc_value = float(comparison.loc[comparison.condition==name,'auroc'].iloc[0])
        ax.plot(fpr,tpr,color=COLORS[name],lw=2,label=f'{LABELS[name]} (AUROC = {auc_value:.4f})')
    ax.set_xlim([0.0,1.0])
    ax.set_ylim([0.0,1.05])
    ax.set_xlabel('False Positive Rate (偽陽性率 / 過剰検知)',fontsize=14)
    ax.set_ylabel('True Positive Rate (真陽性率 / 再現率)',fontsize=14)
    ax.set_title(f'ROC Curve (capsule / Hybrid CNN + FC / seed{SEED})',fontsize=16,fontweight='bold')
    ax.legend(loc='lower right',fontsize=12)
    ax.grid(True,linestyle='--',alpha=0.7)
    fig.tight_layout()
    fig.savefig(RESULT_DIR/'roc_comparison.png',dpi=150)
    plt.show()
    plt.close(fig)
else:
    print('正式結果待ち：この比較表示はまだありません。')

正式結果待ち：この比較表示はまだありません。


## 7. 正常・異常のスコア分布
正常は緑、異常は赤。3条件で共通のビン境界40点と軸範囲を使用し、閾値は表示しません。

In [8]:
if READY:
    joined_scores = np.concatenate([all_scores[name] for name in CONDITIONS])
    lo,hi=float(joined_scores.min()),float(joined_scores.max())
    if hi<=lo: lo-=1e-6;hi+=1e-6
    common_bins = np.linspace(lo,hi,40)
    y = image_table.true_label.to_numpy()
    max_count = max(np.histogram(all_scores[name][y==label],bins=common_bins)[0].max()
                    for name in CONDITIONS for label in [0,1])
    for name in CONDITIONS:
        fig,ax = plt.subplots(figsize=(10,6))
        ax.hist(all_scores[name][y==0],bins=common_bins,color='green',alpha=0.5,
                edgecolor='white',linewidth=0.5,label='正常 (Good)')
        ax.hist(all_scores[name][y==1],bins=common_bins,color='red',alpha=0.5,
                edgecolor='white',linewidth=0.5,label='異常 (Anomaly)')
        ax.set_xlim(common_bins[0],common_bins[-1])
        ax.set_ylim(0,max_count+1)
        ax.set_title(f'異常スコアの分布 ({LABELS[name]} | SSIM Top-{TOP_K})',fontsize=16,fontweight='bold')
        ax.set_xlabel(f'異常スコア (上位{TOP_K}点のSSIM誤差平均)',fontsize=14)
        ax.set_ylabel('画像の数 (Count)',fontsize=14)
        ax.legend(fontsize=12)
        ax.grid(axis='y',linestyle='--',alpha=0.7)
        fig.tight_layout()
        fig.savefig(RESULT_DIR/f'score_distribution_{name}.png',dpi=150)
        plt.show()
        plt.close(fig)
else:
    print('正式結果待ち：この比較表示はまだありません。')

正式結果待ち：この比較表示はまだありません。


## 8. 全画像のスコアと評価記録の保存
正式モデルの重みは変更しません。結果は実行ごとの新規フォルダへ保存します。

In [9]:
if READY:
    score_table=image_table[['relative_path','category_ja','true_label']].copy()
    for name in CONDITIONS: score_table[name]=all_scores[name]
    display(score_table.rename(columns={'relative_path':'テスト内のファイル','category_ja':'種類','true_label':'正解（0正常・1異常）'}).style.format({name:'{:.6f}' for name in CONDITIONS}))
    evaluation_record={'evaluation_datetime_jst':datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
        'training_run_id':RUN_ID,'seed':SEED,'test_counts':counts,'test_manifest_sha256':manifest_hash,
        'inference':'posterior mean; model.eval(); no sampling',
        'preprocessing':'Full RGB -> Resize128 LANCZOS -> Grayscale -> ToTensor; no rotation or padding',
        'ssim_window':11,'top_k':TOP_K,'device':str(DEVICE),'torch_threads':torch.get_num_threads(),
        'training_config':train_config,'metrics':comparison.to_dict('records'),
        'python':platform.python_version(),
        'versions':{v:importlib.metadata.version(v) for v in ['torch','torchvision','numpy','scikit-image','Pillow']},
        'test_manifest':image_records}
    comparison.to_csv(RESULT_DIR/'comparison.csv',index=False)
    score_table.to_csv(RESULT_DIR/'image_scores.csv',index=False)
    (RESULT_DIR/'evaluation_record.json').write_text(json.dumps(evaluation_record,ensure_ascii=False,indent=2)+chr(10),encoding='utf-8')
    assert all(sha256(model_paths[name])==before_model_hashes[name] for name in CONDITIONS)
    print(f'結果を保存しました：{RESULT_DIR}')
else:
    print('正式学習は未実施です。正式モデル・評価結果フォルダは作成していません。')

正式学習は未実施です。正式モデル・評価結果フォルダは作成していません。
